In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "snhupassword"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)



#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))),
    html.Hr(),
    html.Div(
        

    ),
    html.Div(
    [
        html.Div(
            html.B("Summary Statistics"),
            style={
                "fontSize": "22px",
                "marginBottom": "10px",
                "textAlign": "center"
            }
        ),

        html.Div(
            [
                html.Div("Total Animals", id="total-animals", style={
                    "border": "1px solid #ccc",
                    "padding": "15px",
                    "margin": "5px",
                    "width": "25%",
                    "textAlign": "center"
                }),

                html.Div("Unique Breeds", id="unique-breeds", style={
                    "border": "1px solid #ccc",
                    "padding": "15px",
                    "margin": "5px",
                    "width": "25%",
                    "textAlign": "center"
                }),

                html.Div("Most Common Breed", id="mc-breeds", style={
                    "border": "1px solid #ccc",
                    "padding": "15px",
                    "margin": "5px",
                    "width": "25%",
                    "textAlign": "center"
                }),

                html.Div("Average Age", id="average-age", style={
                    "border": "1px solid #ccc",
                    "padding": "15px",
                    "margin": "5px",
                    "width": "25%",
                    "textAlign": "center"
                })
            ],
            id="stats-row",
            style={
                "display": "flex",
                "justifyContent": "space-between",
                "marginBottom": "20px"
            }
        )
    ],
    id="parent-div",
    style={
        "margin": "20px"
    }
),
    html.Div(
        dcc.RadioItems(['Water Rescue', 'Mountain Rescue', 'Disaster Rescue', 'Reset'],
                       id='filter-type')
    ),       
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         sort_action='native',
                         filter_action='native',
        
                         row_selectable = "single",
                         selected_rows=[0],  

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
water_rescue = list(db.read({
            "breed": {"$in": ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }))

mountain_rescue = list(db.read({
            "breed": {"$in": ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 
                              'Siberian Husky', 'Rottweiler']},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }))
disaster_rescue = list(db.read({
            "breed": {"$in": ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                              'Bloodhound', 'Rottweiler']},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }))

dashboard_projection = {
    "animal_type": 1,
    "breed": 1,
    "sex_upon_outcome": 1,
    "age_upon_outcome_in_weeks": 1,
    "outcome_type": 1,
    "name": 1,
    "location_lat": 1,
    "location_long": 1,
    "_id": 0
}

def get_filtered_data(filter_type):
    if filter_type == "Water Rescue":
        filtered_df = water_rescue

    elif filter_type == "Mountain Rescue":
        filtered_df = mountain_rescue

    elif filter_type == "Disaster Rescue":
        filtered_df = disaster_rescue

    else:
        filtered_df = list(db.read({}))

    return filtered_df

def process_filtered_data(filtered_data):
    breed_counts = {}
    outcome_counts = {}
    unique_breeds = set()

    total_animals = 0
    total_age = 0

    for animal in filtered_data:
        total_animals += 1

        breed = animal.get("breed", "Unknown")
        outcome = animal.get("outcome_type", "Unknown")
        age = animal.get("age_upon_outcome_in_weeks", 0)

        if breed in breed_counts:
            breed_counts[breed] += 1
        else:
            breed_counts[breed] = 1

        if outcome in outcome_counts:
            outcome_counts[outcome] += 1
        else:
            outcome_counts[outcome] = 1

        unique_breeds.add(breed)

        if isinstance(age, (int, float)):
            total_age += age

    if total_animals > 0:
        average_age = total_age / total_animals
    else:
        average_age = 0

    return breed_counts, outcome_counts, unique_breeds, total_animals, average_age
    
@app.callback(
    Output('datatable-id', 'data'),
    Output('graph-id', 'children'),
    Output('total-animals', 'children'),
    Output('unique-breeds', 'children'),
    Output('mc-breeds', 'children'),
    Output('average-age', 'children'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    filtered_df = get_filtered_data(filter_type)

    breed_counts, outcome_counts, unique_breeds, total_animals, average_age = process_filtered_data(filtered_df)

    if not filtered_df:
        return [], [], "Total Animals: 0", "Unique Breeds: 0", "Most Common Breed: None", "Average Age: 0 weeks"

    top10 = sorted(
        breed_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )[:10]

    breed_names = [item[0] for item in top10]
    breed_values = [item[1] for item in top10]

    if top10:
        most_common_breed = top10[0][0]
    else:
        most_common_breed = "None"

    graph = dcc.Graph(
        figure=px.pie(
            names=breed_names,
            values=breed_values,
            title="Top Breeds"
        )
    )

    total_animals_text = "Total Animals: " + str(total_animals)
    unique_breeds_text = "Unique Breeds: " + str(len(unique_breeds))
    most_common_breed_text = "Most Common Breed: " + most_common_breed
    average_age_text = "Average Age: " + str(round(average_age, 2)) + " weeks"

    return (
        filtered_df,
        [graph],
        total_animals_text,
        unique_breeds_text,
        most_common_breed_text,
        average_age_text
    )


#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    else:
        return [{
            'if': { 'column_id': i },
            'background_color': '#D2F3FF'
        } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://decimalyear-julymiami-3000.codio.io/proxy/8050/
